In [1]:
pip install datasets tensorflow scikit-learn

In [2]:
import numpy as np
from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [3]:
dataset = load_dataset("dair-ai/emotion")

train_text = dataset["train"]["text"]
train_label = dataset["train"]["label"]

test_text = dataset["test"]["text"]
test_label = dataset["test"]["label"]

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  129kB            

split/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [4]:
max_words = 10000
max_len = 50

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train_text)

X_train = tokenizer.texts_to_sequences(train_text)
X_test = tokenizer.texts_to_sequences(test_text)

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

In [5]:
model = Sequential([
    Embedding(max_words, 64, input_length=max_len),
    LSTM(64),
    Dense(6, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.fit(
    X_train,
    np.array(train_label),
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 17s 35ms/step - accuracy: 0.4826 - loss: 1.3267 - val_accuracy: 0.7009 - val_loss: 0.8160
Epoch 2/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 14s 36ms/step - accuracy: 0.8227 - loss: 0.5131 - val_accuracy: 0.8897 - val_loss: 0.3640
Epoch 3/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 14s 35ms/step - accuracy: 0.9348 - loss: 0.1984 - val_accuracy: 0.9112 - val_loss: 0.2588
Epoch 4/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - accuracy: 0.9685 - loss: 0.1028 - val_accuracy: 0.9047 - val_loss: 0.2801
Epoch 5/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.9775 - loss: 0.0730 - val_accuracy: 0.9100 - val_loss: 0.2700


In [7]:
loss, accuracy = model.evaluate(X_test, np.array(test_label))

print("Test Accuracy:", accuracy)

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9160 - loss: 0.2555
Test Accuracy: 0.9160000085830688


In [13]:
emotion_names = [
    "sadness",
    "joy",
    "love",
    "anger",
    "fear",
    "surprise"
]

text = ["I am feeling very happy today"]

seq = tokenizer.texts_to_sequences(text)
pad = pad_sequences(seq, maxlen=max_len)

prediction = model.predict(pad)

print("Predicted Emotion:",
      emotion_names[np.argmax(prediction)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Predicted Emotion: joy
